# Nonlinear shells

Naghdi shell model with large deformations, discretized by a mixed method.
Joint work with M. Neunteufel ['19, '21], PhD thesis Neunteufel (2021).

A cylindrical tube is clamped at the left rim and pulled apart by the force density $f = (0, \lambda y, 0)$.
The shell is a surface mesh in 3D; the clamped rim is a named edge of the CAD model:

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

In [ ]:
cyl = Cylinder(Axes((0,0,0), X, Y), r=0.4, h=1).faces.Nearest((0.5,0.4,0))
cyl.edges.Min(X).name="left"

mesh = cyl.GenerateMesh(maxh=0.1).Curve(3)
Draw (mesh);

## Unknown fields

* displacement $u \in [H^1(S)]^3$
* linearized rotation $\beta \in H(\operatorname{curl},S)$
* bending moments $\sigma \in H(\operatorname{div} \operatorname{div},S)$

The moments as an extra unknown avoid $C^1$-continuous elements for the bending term (TDNNS method).
All spaces live on the surface, the trace operators give their surface values:

In [ ]:
order = 3

fes1 = HDivDivSurface(mesh, order=order-1)
fes2 = VectorH1(mesh, order=order, dirichlet_bbnd="left")
fes3 = HCurl(mesh, order=order-1, dirichlet_bbnd="left")
fes = fes1 * fes2 * fes3

sigma,u,beta = fes.TrialFunction()
tau,v,delta = fes.TestFunction()

sigma = sigma.Trace()
tau = tau.Trace()
beta = beta.Trace()
delta = delta.Trace()

gradv = Grad(v).Trace()
gradu = Grad(u).Trace()

nsurf = specialcf.normal(3)
t = specialcf.tangential(3)

nel = Cross(nsurf, t)

ngradv = gradv.trans*nsurf
ngradu = gradu.trans*nsurf

sigman = sigma*nel
taun = tau*nel

def tang(u):
    return (u*t)*t

thickness = 0.1

## Energy

The total energy is the sum of

* membrane energy, nonlinear in the displacement $u$, with the surface Green strain $E_{tt} = C_{tt}(u) - P_t$:
$$
\int_S |E_{tt}(u)|^2
$$
* shear energy, with $\beta$ the linearized rotation in the tangential plane:
$$
\int_S | n^T \nabla u - \beta |^2
$$
* bending energy, by the mixed TDNNS method:
$$
t^2 \int_S | \varepsilon_{tt}(\beta) |^2
$$

minus the work of the external force. Nonlinear terms are entered as `Variation` of an energy:

In [ ]:
a = BilinearForm(fes)

bending = -1/thickness**2*InnerProduct(sigma,tau)*ds + \
    (-div(sigma)*delta - div(tau)*beta)*ds + \
    (sigman * tang(delta) + taun*tang(beta))*ds(element_boundary=True)

shear = (ngradu-beta)*(ngradv-delta)*ds

a += (bending+shear).Compile()

Pt = Id(3) - OuterProduct(nsurf, nsurf)
Ft = gradu + Pt
Ctt = Ft.trans * Ft
Ett = Ctt - Pt
a += Variation((InnerProduct(Ett,Ett)*ds).Compile())

factor = Parameter(0.1)
a += Variation((-factor*y*u[1]*ds).Compile())

gfu = GridFunction(fes)

Increase the load step-wise and solve the nonlinear problem by Newton's method.
First and second derivatives of the energy are computed by automatic differentiation.
The color shows the norm of the bending moments $\sigma$ on the deformed shell:

In [ ]:
from ngsolve.solvers import NewtonMinimization

sceneu = Draw(Norm(gfu.components[0]), mesh, deformation=gfu.components[1])

# store displacement and moment norm of each load step for the animation
fesm = H1(mesh, order=2)
gfmom, gfm = GridFunction(fesm, multidim=0), GridFunction(fesm)
gfdisp = GridFunction(fes2, multidim=0)
gfmom.AddMultiDimComponent(gfm.vec)
gfdisp.AddMultiDimComponent(gfu.components[1].vec)

for loadstep in range(4):
    factor.Set (3*(loadstep+1))
    with TaskManager():
        NewtonMinimization (a, gfu, inverse="sparsecholesky", printing=False)
    sceneu.Redraw()
    gfm.Set(Norm(gfu.components[0]), definedon=mesh.Boundaries(".*"))
    gfmom.AddMultiDimComponent(gfm.vec)
    gfdisp.AddMultiDimComponent(gfu.components[1].vec)
    ux, uy, uz = gfu.components[1](mesh(1,0.4,0,BND))
    print (f"load {factor.Get():4.1f},  displacement at free rim: ({ux:6.3f}, {uy:6.3f}, {uz:6.3f})")

The deformation over the load steps, colored by the norm of the bending moments:

In [ ]:
maxmom = gfm.vec.FV().NumPy()[:mesh.nv].max()   # vertex values at last load step
Draw (gfmom, mesh, deformation=gfdisp, interpolate_multidim=True, animate=True,
      min=0, max=maxmom, autoscale=False);